# Notebook 01: Setup & Environment Health Check

Notebook này kiểm tra toàn diện môi trường làm việc của dự án Vietnamese RAG:
1. **Kiểm tra phiên bản các thư viện Python** cốt lõi
2. **Kiểm tra phần cứng** (CPU / CUDA GPU)
3. **Kiểm tra nạp cấu hình hệ thống** (`config.yaml` & biến môi trường `.env`)
4. **Kiểm tra kết nối Vector Database** (Qdrant Docker `http://localhost:6333`)
5. **Kiểm tra kết nối LLM API** (Groq API Key)

## 1. Thiết lập Project Root Path & Imports

In [ ]:
import os
import sys
from pathlib import Path

# Thiết lập thư mục gốc dự án
root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

print(f"📂 Project Root Directory: {root}")

## 2. Kiểm tra Thư viện Python & Phần cứng (Hardware Check)

In [ ]:
import torch
import transformers
import sentence_transformers
import qdrant_client
import pypdf
import docx
import pydantic
import yaml

print("=" * 60)
print("📦 PHIÊN BẢN CÁC THƯ VIỆN CỐT LÕI:")
print(f"  • PyTorch              : {torch.__version__}")
print(f"  • Transformers         : {transformers.__version__}")
print(f"  • Sentence-Transformers: {sentence_transformers.__version__}")
print(f"  • Pydantic             : {pydantic.__version__}")
print(f"  • PyPDF                : {pypdf.__version__}")
print(f"  • Python-Docx          : {docx.__version__}")
print("-" * 60)

# Kiểm tra GPU / CUDA
cuda_available = torch.cuda.is_available()
print(f"⚡ CUDA / GPU khả dụng   : {cuda_available}")
if cuda_available:
    print(f"🎮 Tên thiết bị GPU     : {torch.cuda.get_device_name(0)}")
    print(f"💾 Bộ nhớ GPU VRAM      : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("ℹ️  Hệ thống sẽ chạy Embedding & Reranker trên CPU (Auto-fallback an toàn).")
print("=" * 60)

## 3. Kiểm tra Cấu hình Hệ thống & Biến Môi trường (.env)

In [ ]:
from src.config import get_config

cfg = get_config()

print("=" * 60)
print("⚙️  CẤU HÌNH HỆ THỐNG:")
print(f"  • App Name       : {cfg.app.name} ({cfg.app.env})")
print(f"  • Framework      : {cfg.app.framework}")
print(f"  • Thư mục Raw    : {cfg.data.raw_path}")
print(f"  • Embedding Model: {cfg.embedding.model_name} (dim: {cfg.embedding.dimension})")
print(f"  • Qdrant URL     : {cfg.vector_store.url} -> Collection: {cfg.vector_store.collection_name}")
print(f"  • Reranker Model : {cfg.retrieval.reranker.bge.model_name} (enabled: {cfg.retrieval.reranker.enabled})")
print(f"  • LLM Default    : {cfg.llm.default_provider} -> Model: {cfg.llm.groq.model_name}")
print("-" * 60)

# Kiểm tra secret keys
groq_key = cfg.llm.groq.api_key
if groq_key:
    print(f"🔑 GROQ_API_KEY : Đã tìm thấy (độ dài: {len(groq_key)}, tiền tố: {groq_key[:7]}...)")
else:
    print("❌ GROQ_API_KEY : Chưa được cấu hình trong .env")
print("=" * 60)

## 4. Kiểm tra Kết nối tới Vector Database (Qdrant Docker)

In [ ]:
from qdrant_client import QdrantClient

try:
    client = QdrantClient(
        host=cfg.vector_store.host,
        port=cfg.vector_store.port,
        timeout=5,
    )
    collections_resp = client.get_collections()
    collection_names = [c.name for c in collections_resp.collections]
    
    print("=" * 60)
    print(f"✅ KẾT NỐI QDRANT THÀNH CÔNG tại: {cfg.vector_store.url}")
    print(f"📊 Danh sách Collections hiện có: {collection_names}")
    
    if cfg.vector_store.collection_name in collection_names:
        col_info = client.get_collection(cfg.vector_store.collection_name)
        print(f"📌 Collection '{cfg.vector_store.collection_name}':")
        print(f"   - Số lượng vectors : {col_info.points_count or col_info.indexed_vectors_count}")
        print(f"   - Vector dimension : {col_info.config.params.vectors.size}")
        print(f"   - Distance metric  : {col_info.config.params.vectors.distance}")
    else:
        print(f"⚠️ Collection '{cfg.vector_store.collection_name}' chưa được tạo.")
        print("👉 Hãy chạy: python scripts/run_ingest.py --index để tạo collection và index dữ liệu.")
    print("=" * 60)
except Exception as e:
    print(f"❌ Không thể kết nối tới Qdrant tại {cfg.vector_store.url}: {e}")
    print("👉 Vui lòng mở Docker Desktop và chạy lệnh: docker compose up -d")

## 5. Kiểm tra Kết nối LLM API (Groq Cloud Test)

In [ ]:
import groq

if cfg.llm.groq.api_key:
    try:
        client = groq.Groq(api_key=cfg.llm.groq.api_key)
        response = client.chat.completions.create(
            model=cfg.llm.groq.model_name,
            messages=[
                {"role": "system", "content": "Bạn là một trợ lý AI hữu ích."},
                {"role": "user", "content": "Chào bạn! Trả lời đúng 1 câu ngắn gọn bằng tiếng Việt: Bạn là ai?"},
            ],
            max_tokens=50,
        )
        print("=" * 60)
        print("✅ GỌI GROQ API THÀNH CÔNG!")
        print(f"🤖 Phản hồi: {response.choices[0].message.content}")
        print("=" * 60)
    except Exception as e:
        print(f"❌ Lỗi khi gọi Groq API: {e}")
else:
    print("⚠️ Chưa có GROQ_API_KEY để kiểm tra.")